# Assignment 2 – Exercise 3: Convolutional Autoencoder OOD Detection


In [ ]:
import json, torch, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
ds_raw = load_dataset('blanchon/UC_Merced')['train']
splits_path = Path('../Assignment1/splits.json')
if not splits_path.exists():
    splits_path = Path('splits.json')
with open(splits_path) as f:
    splits = json.load(f)

class UCMercedDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        item = ds_raw[self.indices[idx]]
        img = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

class FIDS30Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = list(Path(root_dir).rglob('*.jpg')) + list(Path(root_dir).rglob('*.png'))
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, -1

tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_ds   = UCMercedDataset(splits['train'], tf)
test_ds    = UCMercedDataset(splits['test'],  tf)
fids30_ds  = FIDS30Dataset('PrepData/FIDS30', tf)
train_loader  = DataLoader(train_ds,  batch_size=32, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_ds,   batch_size=32, shuffle=False, num_workers=0)
fids30_loader = DataLoader(fids30_ds, batch_size=32, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}, Test: {len(test_ds)}, FIDS30: {len(fids30_ds)}')


## Convolutional Autoencoder Architecture


In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Tanh(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = ConvAutoencoder().to(device)
print(f'Autoencoder parameters: {sum(p.numel() for p in ae.parameters()):,}')


## Train Autoencoder (MSE loss, Adam lr=1e-3, 15 epochs)


In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(ae.parameters(), lr=1e-3)
EPOCHS = 15
ae_ckpt = 'autoencoder_best.pth'

if not Path(ae_ckpt).exists():
    best_loss = float('inf')
    train_losses = []
    for epoch in range(1, EPOCHS + 1):
        ae.train()
        epoch_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon = ae(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
        epoch_loss /= len(train_ds)
        train_losses.append(epoch_loss)
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(ae.state_dict(), ae_ckpt)
        print(f'Epoch {epoch:02d}/{EPOCHS} | Loss: {epoch_loss:.6f}', flush=True)
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, marker='o')
    plt.title('Autoencoder Training Loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE')
    plt.tight_layout()
    plt.savefig('ae_training_loss.png', dpi=100)
    plt.show()
else:
    print('Checkpoint found – skipping training.')

ae.load_state_dict(torch.load(ae_ckpt, map_location=device))
ae.eval()
print('Autoencoder ready.')


## Compute Reconstruction MSE (OOD Score)


In [ ]:
def compute_mse_scores(model, loader, device):
    scores = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon = model(x)
            mse = ((recon - x) ** 2).mean(dim=(1, 2, 3))
            scores.extend(mse.cpu().numpy())
    return np.array(scores)

print('Computing MSE for UC Merced test set...')
uc_mse   = compute_mse_scores(ae, test_loader,   device)
print('Computing MSE for FIDS30...')
fids_mse = compute_mse_scores(ae, fids30_loader, device)

threshold_ae = np.percentile(uc_mse, 95)
false_alarm  = (uc_mse   > threshold_ae).mean()
detection    = (fids_mse > threshold_ae).mean()

print(f'Threshold (95th pct of ID MSE): {threshold_ae:.6f}')
print(f'False Alarm Rate  (UC Merced > threshold): {false_alarm:.4f}  ({false_alarm*100:.1f}%)')
print(f'OOD Detection Rate (FIDS30 > threshold):  {detection:.4f}  ({detection*100:.1f}%)')


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(uc_mse,   bins=60, alpha=0.6, color='steelblue', label='UC Merced (In-Dist)', density=True)
plt.hist(fids_mse, bins=60, alpha=0.6, color='tomato',    label='FIDS30 (OOD)',        density=True)
plt.axvline(threshold_ae, color='black', linestyle='--', linewidth=2, label=f'95th pct threshold = {threshold_ae:.5f}')
plt.xlabel('Reconstruction MSE')
plt.ylabel('Density')
plt.title('Autoencoder OOD Detection – MSE Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('ae_ood_histogram.png', dpi=100)
plt.show()


## Visualize Input / Reconstruction Pairs


In [ ]:
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0.0, 1.0)

def show_recon_pairs(model, loader, n=4, title=''):
    model.eval()
    imgs, _ = next(iter(loader))
    with torch.no_grad():
        recons = model(imgs[:n].to(device)).cpu()
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 4))
    for i in range(n):
        axes[0, i].imshow(denorm(imgs[i]).permute(1, 2, 0))
        axes[0, i].set_title('Input');          axes[0, i].axis('off')
        axes[1, i].imshow(denorm(recons[i]).permute(1, 2, 0))
        axes[1, i].set_title('Reconstruction'); axes[1, i].axis('off')
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    safe = title.replace(' ', '_').replace('/', '-')
    plt.savefig(f'recon_{safe}.png', dpi=100)
    plt.show()

show_recon_pairs(ae, test_loader,   n=4, title='UC Merced (In-Distribution)')
show_recon_pairs(ae, fids30_loader, n=4, title='FIDS30 (Out-of-Distribution)')


## Discussion

| Metric | Deep Ensemble | Autoencoder |
|--------|--------------|-------------|
| OOD Detection Rate | (see notebook 2) | (see above) |
| False Alarm Rate | ~5% | ~5% |
| Models trained | 10 | 1 |
| Labels required | Yes | No |
| Training cost | High (10 × 5 epochs) | Low (15 epochs) |

**Which method wins?**  
For drastically different OOD data (aerial vs. fruit), both methods detect FIDS30 reliably. The autoencoder is label-free and computationally cheaper. The deep ensemble models epistemic uncertainty more explicitly and is more robust to near-OOD inputs. For this task the autoencoder is the pragmatic choice; for safety-critical applications the ensemble's calibrated uncertainty estimates justify the cost.
